# India AQI — ML Model Notebook
**Team Hackovators | India Innovates 2026**

This notebook runs all ML models on the AQI dataset:
1. Multivariate Linear Regression
2. Polynomial Regression (degree 2)
3. Z-Score Anomaly Detection
4. Pollution Source Classification
5. 24-Hour AQI Forecasting

**Setup:** `pip install pandas numpy matplotlib scikit-learn seaborn`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split

plt.style.use('dark_background')
print('Libraries loaded.')

## 1. Load Data
Replace `data_template.csv` with your actual dataset.

In [ ]:
# Load dataset — replace with your actual file path
df = pd.read_csv('data_template.csv', comment='#')

# Drop rows with missing key values
df.dropna(subset=['AQI', 'PM2.5', 'PM10', 'NO2'], inplace=True)
df = df[df['AQI'] > 0]

print(f'Records loaded: {len(df)}')
print(f'Cities: {df["City"].nunique()}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Pollutant Distributions', fontsize=14, color='white')

pollutants = ['AQI', 'PM2.5', 'PM10', 'NO2', 'SO2', 'CO']
colors = ['#1d8cf8', '#f97316', '#eab308', '#ef4444', '#a855f7', '#22d3ee']

for ax, col, color in zip(axes.flat, pollutants, colors):
    ax.hist(df[col].dropna(), bins=30, color=color, alpha=0.8, edgecolor='none')
    ax.set_title(col, color=color)
    ax.set_facecolor('#0c1219')

plt.tight_layout()
plt.show()

In [ ]:
# City-wise average AQI
city_avg = df.groupby('City')['AQI'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 5))
bars = plt.bar(city_avg.index, city_avg.values, color='#1d8cf8', alpha=0.85)
plt.axhline(100, color='#eab308', linestyle='--', linewidth=1, label='Moderate threshold')
plt.axhline(200, color='#ef4444', linestyle='--', linewidth=1, label='Unhealthy threshold')
plt.title('Average AQI by City', color='white')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Model 1 — Multivariate Linear Regression
Features: PM10, NO2 → Target: AQI

In [ ]:
features = ['PM10', 'NO2']
target = 'AQI'

X = df[features].fillna(0)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lin_model = LinearRegression()
lin_model.fit(X_train, y_train)
y_pred_lin = lin_model.predict(X_test)

r2_lin = r2_score(y_test, y_pred_lin)
mae_lin = mean_absolute_error(y_test, y_pred_lin)

print('=== Linear Regression ===')
print(f'Intercept : {lin_model.intercept_:.2f}')
print(f'PM10 coef : {lin_model.coef_[0]:.4f}')
print(f'NO2  coef : {lin_model.coef_[1]:.4f}')
print(f'R²        : {r2_lin:.4f}')
print(f'MAE       : ±{mae_lin:.2f} AQI units')

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred_lin, alpha=0.6, color='#1d8cf8', s=30)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=1.5, label='Perfect fit')
plt.xlabel('Actual AQI')
plt.ylabel('Predicted AQI')
plt.title(f'Linear Regression — R²={r2_lin:.3f}')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Model 2 — Polynomial Regression (degree 2)
Feature: PM10 → Target: AQI

In [ ]:
X_poly_raw = df[['PM10']].fillna(0)
y_poly = df['AQI']

poly = PolynomialFeatures(degree=2, include_bias=True)
X_poly = poly.fit_transform(X_poly_raw)

Xp_train, Xp_test, yp_train, yp_test = train_test_split(X_poly, y_poly, test_size=0.2, random_state=42)

poly_model = LinearRegression()
poly_model.fit(Xp_train, yp_train)
y_pred_poly = poly_model.predict(Xp_test)

r2_poly = r2_score(yp_test, y_pred_poly)
mae_poly = mean_absolute_error(yp_test, y_pred_poly)

c = poly_model.coef_
print('=== Polynomial Regression (deg 2) ===')
print(f'AQI = {poly_model.intercept_:.2f} + {c[1]:.4f}*PM10 + {c[2]:.6f}*PM10²')
print(f'R²  : {r2_poly:.4f}')
print(f'MAE : ±{mae_poly:.2f} AQI units')

In [ ]:
pm10_range = np.linspace(df['PM10'].min(), df['PM10'].max(), 200).reshape(-1, 1)
pm10_poly = poly.transform(pm10_range)
aqi_curve = poly_model.predict(pm10_poly)

plt.figure(figsize=(8, 5))
plt.scatter(df['PM10'], df['AQI'], alpha=0.4, color='#eab308', s=20, label='Data')
plt.plot(pm10_range, aqi_curve, color='#f97316', linewidth=2, label='Poly fit (deg 2)')
plt.xlabel('PM10 (μg/m³)')
plt.ylabel('AQI')
plt.title(f'Polynomial Regression — R²={r2_poly:.3f}')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Model 3 — Z-Score Anomaly Detection

In [ ]:
def detect_anomalies(series, threshold=1.8):
    mean = series.mean()
    std  = series.std()
    z    = (series - mean) / std
    return pd.DataFrame({'value': series, 'zscore': z.round(2), 'is_anomaly': z.abs() > threshold})

anom_df = detect_anomalies(df['AQI'])
anomalies = anom_df[anom_df['is_anomaly']]

print(f'Total records : {len(anom_df)}')
print(f'Anomalies     : {len(anomalies)} ({100*len(anomalies)/len(anom_df):.1f}%)')
print(f'Max |z-score| : {anom_df["zscore"].abs().max():.2f}')

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(anom_df['value'].values, color='#1d8cf8', linewidth=0.8, alpha=0.7, label='AQI')
plt.scatter(anomalies.index, anomalies['value'], color='#ef4444', s=40, zorder=5, label=f'Anomalies ({len(anomalies)})')
plt.axhline(anom_df['value'].mean(), color='#22c55e', linestyle='--', linewidth=1, label='Mean')
plt.title('AQI Anomaly Detection (Z-Score, threshold=1.8)')
plt.xlabel('Record index')
plt.ylabel('AQI')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Pollution Source Classification

In [ ]:
def classify_sources(row):
    sources = []
    pm25 = row.get('PM2.5', 0) or 0
    pm10 = row.get('PM10', 0) or 0
    no2  = row.get('NO2',  0) or 0
    so2  = row.get('SO2',  0) or 0
    nh3  = row.get('NH3',  0) or 0

    if pm10 > pm25 * 2.2:
        sources.append('Construction/Road Dust')
    if no2 > 40:
        sources.append('Vehicular Emissions')
    if so2 > 15:
        sources.append('Industrial Discharge')
    if pm25 > 80 and no2 < 35:
        sources.append('Biomass Burning')
    if nh3 > 10:
        sources.append('Agricultural Emissions')
    return ', '.join(sources) if sources else 'Mixed Urban Pollution'

df['PollutionSource'] = df.apply(classify_sources, axis=1)

source_counts = df['PollutionSource'].str.split(', ').explode().value_counts()
print(source_counts)

plt.figure(figsize=(8, 4))
source_counts.plot(kind='barh', color='#1d8cf8', alpha=0.85)
plt.title('Pollution Source Classification')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

## 7. 24-Hour AQI Forecast (Diurnal Pattern)

In [ ]:
def forecast_24h(current_aqi, seed=42):
    np.random.seed(seed)
    hours = np.arange(24)
    # Diurnal multipliers: peaks at morning rush (7-9) and evening rush (17-20)
    multipliers = np.where((hours >= 7) & (hours <= 9), 1.22,
                  np.where((hours >= 17) & (hours <= 20), 1.15,
                  np.where(hours <= 4, 0.85, 1.0)))
    noise = np.random.uniform(-0.05, 0.05, 24)
    forecast = np.maximum(10, np.round(current_aqi * (multipliers + noise))).astype(int)
    return pd.DataFrame({'hour': hours, 'aqi': forecast})

# Example: forecast for average AQI
avg_aqi = int(df['AQI'].mean())
fc = forecast_24h(avg_aqi)

colors_fc = ['#22c55e' if v <= 50 else '#eab308' if v <= 100 else '#f97316' if v <= 150 else '#ef4444' if v <= 200 else '#a855f7' for v in fc['aqi']]

plt.figure(figsize=(12, 4))
plt.bar(fc['hour'], fc['aqi'], color=colors_fc, alpha=0.85, width=0.8)
plt.axhline(avg_aqi, color='white', linestyle='--', linewidth=1, alpha=0.5, label=f'Current AQI: {avg_aqi}')
plt.xticks(range(24), [f'{h:02d}:00' for h in range(24)], rotation=45, ha='right', fontsize=8)
plt.title('24-Hour AQI Forecast (Diurnal Pattern)')
plt.ylabel('AQI')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Peak AQI  : {fc["aqi"].max()} at {fc.loc[fc["aqi"].idxmax(), "hour"]:02d}:00')
print(f'Min  AQI  : {fc["aqi"].min()} at {fc.loc[fc["aqi"].idxmin(), "hour"]:02d}:00')

## 8. Model Summary

In [ ]:
print('=' * 45)
print('         MODEL PERFORMANCE SUMMARY')
print('=' * 45)
print(f'Linear Regression   R²={r2_lin:.3f}  MAE=±{mae_lin:.1f}')
print(f'Polynomial (deg 2)  R²={r2_poly:.3f}  MAE=±{mae_poly:.1f}')
print(f'Anomalies detected  : {len(anomalies)} / {len(df)} records')
print(f'Cities analysed     : {df["City"].nunique()}')
print('=' * 45)